# Creating a Dictionary from All NHC Hurricane Archive Data

In [1]:
from herbie import Herbie
import numpy as np
import pandas as pd
import csv

In [2]:
filename = 'hurdat2-1851-2023-051124.txt'
missingVal = -999
basin='AL'
columnHeaders = ['id', 'name', 'entries', 'date', 'time', 'record_identifier', 'status', 'lat', 'lon', 'vmax', 'pres', '34ne', '34se', '34sw', '34nw', '50ne', '50se', '50sw', '50nw', '64ne', '64se', '64sw', '64nw', 'rmax']

In [3]:
# Reformatting input file to csv and creating a new file named 'data.csv'
with open(filename, 'r') as f_in, open('all_data.csv', 'w') as f_out:
    tmp = ""
    for line in f_in:
        line = line.replace(' ', '')  
        if line.startswith(basin):
            tmp = line.strip()  
        else:
            line = tmp + line  
            f_out.write(line)  

In [4]:
data = pd.read_csv('all_data.csv', header=None, names=columnHeaders)
data.replace(-999, np.nan, inplace=True) # Remove to keep -999 values
data

,id,name,entries,date,time,record_identifier,status,lat,lon,vmax,...,34nw,50ne,50se,50sw,50nw,64ne,64se,64sw,64nw,rmax
0,AL011851,UNNAMED,14,18510625,0,NaN,HU,28.0N,94.8W,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AL011851,UNNAMED,14,18510625,600,NaN,HU,28.0N,95.4W,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AL011851,UNNAMED,14,18510625,1200,NaN,HU,28.0N,96.0W,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AL011851,UNNAMED,14,18510625,1800,NaN,HU,28.1N,96.5W,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AL011851,UNNAMED,14,18510625,2100,L,HU,28.2N,96.8W,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54744,AL212023,TWENTY-ONE,6,20231023,1800,NaN,TD,11.5N,83.2W,25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0
54745,AL212023,TWENTY-ONE,6,20231024,0,NaN,TD,12.2N,83.4W,25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0
54746,AL212023,TWENTY-ONE,6,20231024,130,L,TD,12.4N,83.5W,25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0
54747,AL212023,TWENTY-ONE,6,20231024,600,NaN,TD,13.0N,83.8W,25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0


In [5]:
# Convert latitude
data['lat'] = data['lat'].str[:-1].astype(float)


# Convert longitude
data['lon'] = data['lon'].apply(lambda x: '-' + x if not x.startswith('-') else x)
data['lon'] = data['lon'].str[:-1].astype(float)

data

,id,name,entries,date,time,record_identifier,status,lat,lon,vmax,...,34nw,50ne,50se,50sw,50nw,64ne,64se,64sw,64nw,rmax
0,AL011851,UNNAMED,14,18510625,0,NaN,HU,28.0,-94.8,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AL011851,UNNAMED,14,18510625,600,NaN,HU,28.0,-95.4,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AL011851,UNNAMED,14,18510625,1200,NaN,HU,28.0,-96.0,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AL011851,UNNAMED,14,18510625,1800,NaN,HU,28.1,-96.5,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AL011851,UNNAMED,14,18510625,2100,L,HU,28.2,-96.8,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54744,AL212023,TWENTY-ONE,6,20231023,1800,NaN,TD,11.5,-83.2,25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0
54745,AL212023,TWENTY-ONE,6,20231024,0,NaN,TD,12.2,-83.4,25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0
54746,AL212023,TWENTY-ONE,6,20231024,130,L,TD,12.4,-83.5,25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0
54747,AL212023,TWENTY-ONE,6,20231024,600,NaN,TD,13.0,-83.8,25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,60.0


In [6]:
data.to_csv('all_data.csv', index=False)

In [28]:
# Creating a dictionary - one key for each hurricane - alternate hurDictset
hurDict = {}
newBasin = False

with open('all_data.csv', 'r') as f_in:
    reader = csv.reader(f_in)
    next(reader)
    for line in reader:
        if line[0] not in hurDict.keys():
            while len(line[4]) < 4: line[4] = '0' + line[4]
            currDate = pd.to_datetime(line[3] + line[4],format='%Y%m%d%H%M')
            hurDict[line[0]] = [line[1], line[3] + line[4], [(currDate-currDate)/pd.Timedelta('1 hour')], [[line[7], line[8]]], [line[9]], [line[10]], [line[11:15]], [line[15:19]], [line[19:23]], [line[23]]]
            
        else:
            while len(line[4]) < 4: line[4] = '0' + line[4]
            date = pd.to_datetime(line[3] + line[4],format='%Y%m%d%H%M')

            if newBasin:
                currDate = date
                newBasin = False
            hurDict[line[0]][2].append((date - currDate)/pd.Timedelta('1 hour')) # Date to first numpy array
            hurDict[line[0]][3].append([line[7], line[8]]) # Lat and Lon to second numpy array
            hurDict[line[0]][4].append(line[9]) # vmax to third numpy array
            hurDict[line[0]][5].append(line[10]) # pres to fourth numpy array
            hurDict[line[0]][6].append(line[11:15]) # 34ne, 34se, 34sw, 34nw to fifth numpy array
            hurDict[line[0]][7].append(line[15:19]) # 50ne, 50se, 50sw, 50nw to sixth numpy array
            hurDict[line[0]][8].append(line[19:23]) # 64ne, 64se, 64sw, 64nw to seventh numpy array
            hurDict[line[0]][9].append(line[23]) # rmax to eighth numpy array

In [29]:
for hur in hurDict.keys():
    hurDict[hur][2] = [float(x) if x != '' else np.nan for x in hurDict[hur][2]]
    hurDict[hur][3] = [[float(x) if x != '' else np.nan for x in arr] for arr in hurDict[hur][3]]
    hurDict[hur][4] = [float(x) if x != '' else np.nan for x in hurDict[hur][4]]
    hurDict[hur][5] = [float(x) if x != '' else np.nan for x in hurDict[hur][5]]
    hurDict[hur][6] = [[float(x) if x != '' else np.nan for x in arr] for arr in hurDict[hur][6]]
    hurDict[hur][7] = [[float(x) if x != '' else np.nan for x in arr] for arr in hurDict[hur][7]]
    hurDict[hur][8] = [[float(x) if x != '' else np.nan for x in arr] for arr in hurDict[hur][8]]
    hurDict[hur][9] = [float(x) if x != '' else np.nan for x in hurDict[hur][9]]

    hurDict[hur].insert(2, (pd.to_datetime(hurDict[hur][1]) + pd.Timedelta(hurDict[hur][2][-1], unit='hour')).strftime('%Y%m%d%H%M'))



In [30]:
df = pd.DataFrame(hurDict)
cols = ['name', 'start', 'end', 'time', 'trajectory', 'vmax', 'pres', '34', '50', '64', 'rmax']
headers = {}
for i in range(len(cols)):
    headers[i] = cols[i]
df = df.T.rename(columns=headers)

print(f'Number of hurricane tracks available: {df.index.nunique()}')
df

Number of hurricane tracks available: 1973


,name,start,end,time,trajectory,vmax,pres,34,50,64,rmax
AL011851,UNNAMED,185106250000,185106280000,"[0.0, 6.0, 12.0, 18.0, 21.0, 24.0, 30.0, 36.0,...","[[28.0, -94.8], [28.0, -95.4], [28.0, -96.0], ...","[80.0, 80.0, 80.0, 80.0, 80.0, 70.0, 60.0, 60....","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[[nan, nan, nan, nan], [nan, nan, nan, nan], [...","[[nan, nan, nan, nan], [nan, nan, nan, nan], [...","[[nan, nan, nan, nan], [nan, nan, nan, nan], [...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
AL021851,UNNAMED,185107051200,185107051200,[0.0],"[[22.2, -97.6]]",[80.0],[nan],"[[nan, nan, nan, nan]]","[[nan, nan, nan, nan]]","[[nan, nan, nan, nan]]",[nan]
AL031851,UNNAMED,185107101200,185107101200,[0.0],"[[12.0, -60.0]]",[50.0],[nan],"[[nan, nan, nan, nan]]","[[nan, nan, nan, nan]]","[[nan, nan, nan, nan]]",[nan]
AL041851,UNNAMED,185108160000,185108271800,"[0.0, 6.0, 12.0, 18.0, 24.0, 30.0, 36.0, 42.0,...","[[13.4, -48.0], [13.7, -49.5], [14.0, -51.0], ...","[40.0, 40.0, 50.0, 50.0, 60.0, 60.0, 70.0, 70....","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[[nan, nan, nan, nan], [nan, nan, nan, nan], [...","[[nan, nan, nan, nan], [nan, nan, nan, nan], [...","[[nan, nan, nan, nan], [nan, nan, nan, nan], [...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
AL051851,UNNAMED,185109130000,185109161800,"[0.0, 6.0, 12.0, 18.0, 24.0, 30.0, 36.0, 42.0,...","[[32.5, -73.5], [32.5, -73.5], [32.5, -73.5], ...","[50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50.0, 50....","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[[nan, nan, nan, nan], [nan, nan, nan, nan], [...","[[nan, nan, nan, nan], [nan, nan, nan, nan], [...","[[nan, nan, nan, nan], [nan, nan, nan, nan], [...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ..."
...,...,...,...,...,...,...,...,...,...,...,...
AL172023,PHILIPPE,202309230600,202310060600,"[0.0, 6.0, 12.0, 18.0, 24.0, 30.0, 36.0, 42.0,...","[[15.5, -36.6], [15.6, -38.0], [15.7, -39.1], ...","[30.0, 30.0, 35.0, 40.0, 45.0, 45.0, 45.0, 45....","[1007.0, 1007.0, 1005.0, 1003.0, 1001.0, 1000....","[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [...","[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [...","[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [...","[70.0, 70.0, 60.0, 50.0, 50.0, 40.0, 40.0, 40...."
AL182023,RINA,202309280600,202310020000,"[0.0, 6.0, 12.0, 18.0, 24.0, 30.0, 36.0, 42.0,...","[[15.6, -44.5], [16.9, -45.1], [17.7, -45.8], ...","[35.0, 35.0, 35.0, 40.0, 45.0, 45.0, 45.0, 45....","[1005.0, 1004.0, 1004.0, 1002.0, 999.0, 999.0,...","[[60.0, 50.0, 0.0, 0.0], [80.0, 60.0, 0.0, 0.0...","[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [...","[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [...","[60.0, 50.0, 50.0, 50.0, 40.0, 40.0, 40.0, 40...."
AL192023,SEAN,202310101800,202310161800,"[0.0, 6.0, 12.0, 18.0, 24.0, 30.0, 36.0, 42.0,...","[[9.6, -30.2], [9.8, -31.4], [10.1, -32.6], [1...","[30.0, 35.0, 35.0, 35.0, 30.0, 30.0, 30.0, 35....","[1007.0, 1006.0, 1006.0, 1006.0, 1007.0, 1007....","[[0.0, 0.0, 0.0, 0.0], [80.0, 0.0, 0.0, 0.0], ...","[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [...","[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [...","[80.0, 80.0, 80.0, 80.0, 60.0, 40.0, 40.0, 40...."
AL202023,TAMMY,202310181800,202310311800,"[0.0, 6.0, 12.0, 18.0, 24.0, 30.0, 36.0, 42.0,...","[[12.9, -51.0], [13.0, -52.5], [13.2, -54.0], ...","[35.0, 35.0, 45.0, 50.0, 50.0, 50.0, 50.0, 60....","[1007.0, 1006.0, 1004.0, 1004.0, 1002.0, 1001....","[[120.0, 0.0, 0.0, 0.0], [120.0, 0.0, 0.0, 60....","[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [...","[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [...","[80.0, 60.0, 60.0, 50.0, 50.0, 50.0, 50.0, 40...."


{'AL011851': ['UNNAMED',
  '185106250000',
  [0.0,
   6.0,
   12.0,
   18.0,
   21.0,
   24.0,
   30.0,
   36.0,
   42.0,
   48.0,
   54.0,
   60.0,
   66.0,
   72.0],
  [[28.0, -94.8],
   [28.0, -95.4],
   [28.0, -96.0],
   [28.1, -96.5],
   [28.2, -96.8],
   [28.2, -97.0],
   [28.3, -97.6],
   [28.4, -98.3],
   [28.6, -98.9],
   [29.0, -99.4],
   [29.5, -99.8],
   [30.0, -100.0],
   [30.5, -100.1],
   [31.0, -100.2]],
  [80.0,
   80.0,
   80.0,
   80.0,
   80.0,
   70.0,
   60.0,
   60.0,
   50.0,
   50.0,
   40.0,
   40.0,
   40.0,
   40.0],
  [nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan],
  [[nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan],
   [nan, nan, nan, nan]],
  [[nan, nan, nan, nan],

In [76]:
import json 
  
with open('all_data_dict_NaN.txt', 'w') as dictFile: 
     dictFile.write(json.dumps(hurDict))